<img src="https://hilpisch.com/tce_logo.png" alt="The Crypto Engineer" width="25%" align="right" border="0"><br>


# Bitcoin Foundations
## Chapter 3 – UTXOs and the Transaction Model

&copy; Dr. Yves J. Hilpisch | The Python Quants GmbH<br>
The Crypto Engineer | AI-Powered by GPT 5.1.


This lab models a tiny UTXO set using Python classes and dictionaries. By stepping through `apply_tx`, you see how inputs remove existing outputs and how new outputs get added—mirroring how a real node evolves its global UTXO set as it processes transactions.

In [1]:
from dataclasses import dataclass  # lightweight data containers
from typing import Dict, List  # type hints for clarity


@dataclass
class UTXO:  # representation of one unspent output
    txid: str  # transaction identifier
    index: int  # output index within transaction
    value: int  # value in satoshis
    owner: str  # simplified owner label for this toy model


UTXOSet = Dict[tuple, UTXO]  # maps (txid, index) to UTXO


def apply_tx(utxos: UTXOSet, inputs, outputs) -> None:  # mutate UTXO set in place
    total_in = 0  # track sum of input values
    for txid, idx in inputs:  # each input references a prior UTXO
        utxo = utxos.pop((txid, idx))  # remove UTXO, raising if missing
        total_in += utxo.value  # add its value to total
    total_out = sum(v for _, v, _ in outputs)  # sum of output values
    assert total_in >= total_out  # enforce conservation with non-negative fee
    for new_txid, value, owner in outputs:  # create new UTXOs
        key = (new_txid, 0)  # toy model: one output per tx
        utxos[key] = UTXO(new_txid, 0, value, owner)  # insert into UTXO set


With the UTXO set and update function in place, you can now play through a concrete "Alice pays Bob" transaction. Watching the before/after state helps make the conservation-of-value and double-spend rules feel much more tangible than in pure prose.

In [2]:
utxos: UTXOSet = {}  # start with an empty set

# initial coin: tx0 -> Alice with 1_000_000 sat
utxos[("tx0", 0)] = UTXO("tx0", 0, 1_000_000, "Alice")  # one UTXO for Alice

inputs = [("tx0", 0)]  # Alice will spend this output
outputs = [("tx1", 300_000, "Bob")]  # Bob receives 0.003 BTC

apply_tx(utxos, inputs, outputs)  # apply transaction to UTXO set

for key, utxo in utxos.items():  # inspect resulting UTXO set
    print(key, utxo)


('tx1', 0) UTXO(txid='tx1', index=0, value=300000, owner='Bob')


<img src="https://hilpisch.com/tce_logo.png" alt="The Crypto Engineer" width="25%" align="right" border="0"><br>
